# 03 — Final Data Cleaning & Export

**Run in VS Code.** Keep `shopping_data.csv` in the same folder as this notebook.

This notebook creates a separate file named `shopping_data_cleaned.csv`. The original CSV is not overwritten.

Cleaning choices used here:
- Remove repeated `Order_Id` values, keeping the first copy. In this source, the repeated order IDs are duplicate records.
- Trim whitespace in text columns and standardize city capitalization.
- Convert date and numeric fields to appropriate types.
- Treat discounts outside 0–50 as invalid and replace them using the median valid discount.
- Fill missing numeric values with their column median, missing `Payment_Mode` with its mode, and missing dates with the median valid date.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_FILE = Path("shopping_data.csv")
OUTPUT_FILE = Path("shopping_data_cleaned.csv")

if not DATA_FILE.exists():
    raise FileNotFoundError(
        "shopping_data.csv was not found. Put it in the same folder as this notebook."
    )

df = pd.read_csv(DATA_FILE)
print("Original dataset shape:", df.shape)
display(df.head())

## 1. Remove duplicate orders

In [ ]:
rows_before = len(df)
df = df.drop_duplicates(subset="Order_Id", keep="first").copy()
print("Duplicate rows removed:", rows_before - len(df))
print("Shape after duplicate removal:", df.shape)

## 2. Standardize text columns

In [ ]:
text_columns = df.select_dtypes(include=["object"]).columns

for column in text_columns:
    df[column] = df[column].astype("string").str.strip()

# Standardize city names (for example, 'mumbai' becomes 'Mumbai').
df["City"] = df["City"].str.title()

# Convert blank strings to proper missing values.
df = df.replace(r"^\\s*$", pd.NA, regex=True)

print("City values after standardization:")
display(df["City"].value_counts(dropna=False))

## 3. Convert date and numeric columns

In [ ]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")

numeric_columns = [
    "Customer_Age",
    "Items_In_Order",
    "Time_On_Site_Min",
    "Discount_Percent",
    "Order_Value_Rs"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

print("Date values missing or not understood:", df["Order_Date"].isna().sum())

## 4. Handle invalid discounts and missing values

In [ ]:
# Treat discounts outside the chosen 0–50% range as invalid.
valid_discount = df["Discount_Percent"].between(0, 50)
invalid_discount_count = int((df["Discount_Percent"].notna() & ~valid_discount).sum())
df.loc[~valid_discount, "Discount_Percent"] = np.nan

# Fill numeric missing values with each column's median.
for column in ["Customer_Age", "Time_On_Site_Min", "Discount_Percent"]:
    median_value = df[column].median()
    if pd.isna(median_value):
        raise ValueError(f"Cannot calculate a median for {column}: no valid values.")
    df[column] = df[column].fillna(median_value)

# Fill missing payment modes with the most frequent value.
payment_mode = df["Payment_Mode"].mode(dropna=True)
if not payment_mode.empty:
    df["Payment_Mode"] = df["Payment_Mode"].fillna(payment_mode.iloc[0])

# Fill missing dates with the middle valid date.
valid_dates = df["Order_Date"].dropna().sort_values()
if df["Order_Date"].isna().any():
    if valid_dates.empty:
        raise ValueError("All dates are missing or invalid; cannot impute a date.")
    middle_date = valid_dates.iloc[len(valid_dates) // 2]
    df["Order_Date"] = df["Order_Date"].fillna(middle_date)

print("Invalid discounts corrected:", invalid_discount_count)
print("Missing values after these steps:")
display(df.isna().sum().to_frame("Missing_Count"))

## 5. Check remaining missing values and finalize types

In [ ]:
# The supplied dataset has no missing values in these fields after the steps above.
# If a different source file is used, report any remaining missing values rather than
# silently inventing category values.
remaining_missing = df.isna().sum()
display(remaining_missing.to_frame("Missing_Count"))

# Use nullable integer types for whole-number columns.
for column in ["Customer_Age", "Items_In_Order", "Order_Value_Rs"]:
    df[column] = df[column].round().astype("Int64")

df["Time_On_Site_Min"] = df["Time_On_Site_Min"].round(1)
df["Discount_Percent"] = df["Discount_Percent"].round(1)

# Preserve the original dataset's column order.
df = df[
    [
        "Order_Id", "Order_Date", "City", "Product_Category", "Payment_Mode",
        "Customer_Type", "Customer_Age", "Items_In_Order", "Time_On_Site_Min",
        "Discount_Percent", "Order_Value_Rs"
    ]
]

## 6. Validate the cleaned dataset

In [ ]:
print("Final shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate Order_Id values:", int(df["Order_Id"].duplicated().sum()))
print("Discounts outside 0–50:", int((~df["Discount_Percent"].between(0, 50)).sum()))

display(df.head())
display(df.describe(include="all").T)

assert df["Order_Id"].is_unique, "Order_Id values are not unique."
assert df.isna().sum().sum() == 0, "Some missing values remain."
assert df["Discount_Percent"].between(0, 50).all(), "Some discounts are outside 0–50%." 

## 7. Export the cleaned CSV

In [ ]:
df.to_csv(OUTPUT_FILE, index=False, date_format="%Y-%m-%d")

print("Cleaned CSV created successfully:")
print(OUTPUT_FILE.resolve())
print("File size:", OUTPUT_FILE.stat().st_size, "bytes")

## 8. Optional: open the output file from VS Code

In [ ]:
# This prints the full path. In VS Code Explorer, click shopping_data_cleaned.csv
# to inspect the generated file.
print("Open this file in VS Code:", OUTPUT_FILE.resolve())